# ABIDE Functional Connectivity Analysis

Exploratory data analysis, CC200 connectivity features, within-site-balanced cross-validation, and leave-one-site-out evaluation.

## Load participant metadata

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Expected folder layout: data/participants.csv and data/timeseries/*.1D
DATA_DIR = Path("data")
PARTICIPANTS_PATH = DATA_DIR / "participants.csv"
TIMESERIES_DIR = DATA_DIR / "timeseries"

if not PARTICIPANTS_PATH.exists():
    raise FileNotFoundError(f"Missing {PARTICIPANTS_PATH}. Place your data in the data/ folder.")
participants = pd.read_csv(PARTICIPANTS_PATH)
participants.head()

## Dataset checks and exploratory analysis

In [ ]:
# Overall subject count
print("Total subjects:", participants["subject_id"].nunique())

# Diagnosis distribution
print("\nDiagnosis distribution:")
print(participants["diagnosis"].value_counts())

In [ ]:
# Site distribution
print("Site distribution:")
print(participants["site"].value_counts())

In [ ]:
# Diagnosis distribution within each site
pd.crosstab(participants["site"], participants["diagnosis"])

In [ ]:
import numpy as np

# Check number of time points by site
timepoint_info = []

for _, row in participants.iterrows():
    subject_id = row["subject_id"]
    site = row["site"]

    file_path = TIMESERIES_DIR / f"{subject_id}_rois_cc200.1D"
    ts = np.loadtxt(file_path)

    timepoint_info.append({
        "subject_id": subject_id,
        "site": site,
        "n_timepoints": ts.shape[0]
    })

timepoint_df = pd.DataFrame(timepoint_info)

timepoint_summary = timepoint_df.groupby("site")["n_timepoints"].agg(
    ["count", "min", "max", "mean"]
)

timepoint_summary

In [ ]:
# Age summary
participants["age"].describe()

In [ ]:
# Mean framewise displacement summary
participants["mean_fd"].describe()

In [ ]:
from scipy.stats import ttest_ind

asd = participants[participants["diagnosis"] == "ASD"]
tdc = participants[participants["diagnosis"] == "TDC"]

# Age: group means and Welch's t-test
print("Age means:")
print(participants.groupby("diagnosis")["age"].mean())

age_t, age_p = ttest_ind(
    asd["age"],
    tdc["age"],
    equal_var=False
)

print(f"\nWelch's t-test: t = {age_t:.3f}, p = {age_p:.3f}")

In [ ]:
# Sex distribution
sex_table = pd.crosstab(
    participants["diagnosis"],
    participants["sex"]
)

print(sex_table)

In [ ]:
# Mean FD: group means and Welch's t-test
print("Mean FD means:")
print(participants.groupby("diagnosis")["mean_fd"].mean())

fd_t, fd_p = ttest_ind(
    asd["mean_fd"],
    tdc["mean_fd"],
    equal_var=False
)

print(f"\nWelch's t-test: t = {fd_t:.3f}, p = {fd_p:.3f}")

In [ ]:
import matplotlib.pyplot as plt

asd_fd = participants.loc[participants["diagnosis"] == "ASD", "mean_fd"]
tdc_fd = participants.loc[participants["diagnosis"] == "TDC", "mean_fd"]

plt.figure(figsize=(6, 4))

plt.boxplot(
    [asd_fd, tdc_fd],
    tick_labels=["ASD", "TDC"]
)

plt.ylabel("Mean Framewise Displacement (mm)")
plt.xlabel("Diagnosis")
plt.title("Head Motion by Diagnostic Group")

plt.show()

## Functional connectivity features

In [ ]:
files = sorted(TIMESERIES_DIR.glob("*.1D"))
if not files:
    raise FileNotFoundError(f"No .1D time-series files found in {TIMESERIES_DIR}")
ts_example = np.loadtxt(files[0])
print("Example time-series shape:", ts_example.shape)

In [ ]:
# Compute FC features for all participants

fc_vectors = []
fc_matrices = []

# Indices of the upper triangle, excluding diagonal
upper_idx = np.triu_indices(200, k=1)

for subject_id in participants["subject_id"]:
    
    # Load this subject's time series
    file_path = TIMESERIES_DIR / f"{subject_id}_rois_cc200.1D"
    ts = np.loadtxt(file_path)
    
    if ts.ndim != 2 or ts.shape[1] != 200:
        raise ValueError(f"Unexpected time-series shape for {subject_id}: {ts.shape}; expected (timepoints, 200)")
    if not np.isfinite(ts).all():
        raise ValueError(f"Non-finite time-series values for {subject_id}")

    # Pearson correlation
    fc = np.corrcoef(ts, rowvar=False)
    
    if not np.isfinite(fc).all():
        raise ValueError(f"Undefined correlations for {subject_id}; check constant ROI signals")

    # Remove diagonal
    np.fill_diagonal(fc, 0)
    
    # Avoid numerical problems if a correlation is exactly +1 or -1
    fc = np.clip(fc, -0.999999, 0.999999)
    
    # Fisher-z transform
    fc_z = np.arctanh(fc)
    
    # Save full matrix for later visualization
    fc_matrices.append(fc_z)
    
    # Save upper triangle as features
    fc_vectors.append(fc_z[upper_idx])

# Convert lists to NumPy arrays
fc_matrices = np.array(fc_matrices)
X = np.array(fc_vectors)

print("FC matrices shape:", fc_matrices.shape)
print("Feature matrix shape:", X.shape)

In [ ]:
# Average FC across all participants in Fisher-z space
mean_fc_z = fc_matrices.mean(axis=0)

# Convert the averaged Fisher-z values back to Pearson r for visualization
mean_fc_r = np.tanh(mean_fc_z)

print("Mean FC matrix shape:", mean_fc_r.shape)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 6))
plt.imshow(mean_fc_r, vmin=-1, vmax=1, aspect="auto")
plt.colorbar(label="Mean Pearson Correlation")
plt.xlabel("ROI")
plt.ylabel("ROI")
plt.title("Mean Functional Connectivity Matrix")
plt.show()

## Repeated stratified cross-validation

In [ ]:
y = participants["diagnosis"].map({"TDC": 0, "ASD": 1}).values

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Class counts:", np.bincount(y))

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RepeatedStratifiedKFold

# Logistic regression pipeline
model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        C=1.0,
        solver="liblinear",
        max_iter=1000
    ))
])

# Joint stratification by site and diagnosis
strata = participants["site"] + "_" + participants["diagnosis"]

print("Strata counts:")
print(strata.value_counts())

# Repeated stratified 5-fold CV
rskf = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=10,
    random_state=42
)

# Generate splits using site + diagnosis strata
cv_splits = list(rskf.split(X, strata))

print("\nNumber of CV splits:", len(cv_splits))

In [ ]:
from sklearn.model_selection import cross_validate

scores = cross_validate(
    model,
    X,
    y,
    cv=cv_splits,
    scoring=["roc_auc", "balanced_accuracy"]
)

In [ ]:
# Average the 5 folds within each repetition
auc_by_repeat = scores["test_roc_auc"].reshape(10, 5).mean(axis=1)
bal_acc_by_repeat = scores["test_balanced_accuracy"].reshape(10, 5).mean(axis=1)

print("ROC-AUC by repetition:", auc_by_repeat)
print("Balanced accuracy by repetition:", bal_acc_by_repeat)

print("\nRepetition-level summary:")
print(f"Mean ROC-AUC: {auc_by_repeat.mean():.3f}")
print(f"SD ROC-AUC: {auc_by_repeat.std(ddof=1):.3f}")

print(f"Mean balanced accuracy: {bal_acc_by_repeat.mean():.3f}")
print(f"SD balanced accuracy: {bal_acc_by_repeat.std(ddof=1):.3f}")

## Leave-one-site-out evaluation

In [ ]:
sites = participants["site"].values

print(np.unique(sites))

In [ ]:
from sklearn.metrics import roc_auc_score, balanced_accuracy_score

site_results = []

for test_site in np.unique(sites):

    # Split into training sites and one held-out test site
    train_idx = sites != test_site
    test_idx = sites == test_site

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # Train model only on the two training sites
    model.fit(X_train, y_train)

    # Predictions
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    # Metrics
    auc = roc_auc_score(y_test, y_prob)
    bal_acc = balanced_accuracy_score(y_test, y_pred)

    site_results.append([test_site, auc, bal_acc])

    print(
        f"Held-out site: {test_site} | "
        f"ROC-AUC: {auc:.3f} | "
        f"Balanced accuracy: {bal_acc:.3f}"
    )

In [ ]:
site_results_array = np.array([[r[1], r[2]] for r in site_results])

print("Mean cross-site ROC-AUC:", site_results_array[:, 0].mean())
print("Mean cross-site balanced accuracy:", site_results_array[:, 1].mean())